# Objective

This notebook introduces the first asynchronous, multi-threaded stage of the profiling pipeline.

The goal is to decouple event production from event processing using a thread-safe bounded queue.

In the previous notebook, events were processed synchronously:

```text
SyntheticEventSource
        ↓
   ProfileEvent
        ↓
Serializer / Sink
```

In this notebook, the pipeline becomes:

```text
Producer Thread
      ↓
BoundedQueue<ProfileEvent>
      ↓
Consumer Thread
      ↓
Serializer / Sink
```

This design allows event generation and event processing to run independently.

The notebook focuses on the following objectives:

- Implement a reusable `BoundedQueue<T>` template.
- Protect shared queue state using `std::mutex`.
- Coordinate producers and consumers using `std::condition_variable`.
- Prevent unbounded memory growth by limiting queue capacity.
- Run event production and consumption on separate threads.
- Use `std::jthread` and `std::stop_token` for modern C++20 thread management.
- Implement a clean shutdown mechanism so blocked threads can exit safely.
- Validate the pipeline under different producer and consumer speeds.

---

## Why a Queue Is Needed

In a real profiling system, event production and event processing may run at different speeds.

For example:

```text
GPU / Profiler
   produces events quickly
          ↓
        Queue
          ↓
Network / Storage
   processes events more slowly
```

Without an intermediate queue, a slow consumer could directly block the event producer.

A queue provides temporary buffering and allows both sides to operate asynchronously.

---

## Why the Queue Is Bounded

An unbounded queue can continue growing if the producer is faster than the consumer.

For example:

```text
Producer: 1,000,000 events/s
Consumer:   300,000 events/s
```

The queue would continuously accumulate events:

```text
queue size
   ↑
   ↑
   ↑
   ↑
memory exhaustion
```

A bounded queue introduces a fixed capacity:

```text
Producer
   ↓
┌───────────────────┐
│   BoundedQueue    │
│ capacity = N      │
└───────────────────┘
   ↓
Consumer
```

When the queue becomes full, the producer must wait until space becomes available.

This creates basic backpressure and prevents uncontrolled memory growth.

---

## Threading Model

The initial implementation uses two threads:

```text
Producer Thread
    |
    | next_event()
    v
BoundedQueue<ProfileEvent>
    |
    | pop()
    v
Consumer Thread
    |
    v
ConsoleSink / Serializer
```

The producer is responsible for:

```text
read/generate event
        ↓
push event into queue
```

The consumer is responsible for:

```text
wait for event
        ↓
remove event from queue
        ↓
process event
```

---

## Synchronization

Because the producer and consumer access the same queue concurrently, the shared state must be protected.

The implementation will use:

```text
std::mutex
std::condition_variable
```

The mutex protects access to the queue and its state.

Condition variables allow threads to sleep efficiently while waiting for a condition.

The producer waits when:

```text
queue is full
```

The consumer waits when:

```text
queue is empty
```

Conceptually:

```text
Producer
   |
   | queue full?
   v
wait on condition_variable
   |
space available
   v
push event
```

and:

```text
Consumer
   |
   | queue empty?
   v
wait on condition_variable
   |
event available
   v
pop event
```

---

## Modern C++20 Thread Management

The pipeline will use:

```cpp
std::jthread
```

instead of manually managing raw `std::thread` objects where possible.

`std::jthread` provides:

- automatic joining
- cooperative cancellation support
- integration with `std::stop_token`

The consumer can therefore periodically check:

```cpp
stop_token.stop_requested()
```

and exit cleanly when shutdown is requested.

---

## Graceful Shutdown

Thread shutdown is an important part of a production-style asynchronous system.

A consumer may be sleeping while waiting for data:

```text
consumer
   ↓
condition_variable::wait()
```

If the program simply exits without notifying it, the thread may remain blocked.

The queue will therefore support a closed state.

Conceptually:

```text
Producer finishes
      ↓
queue.close()
      ↓
wake waiting consumers
      ↓
consumer drains remaining events
      ↓
consumer exits
```

This allows the pipeline to terminate without deadlocks or abandoned work.

---

## Expected Architecture

By the end of this notebook, the pipeline should look like:

```text
┌──────────────────────┐
│ SyntheticEventSource │
└──────────┬───────────┘
           │
           │ producer thread
           ▼
┌──────────────────────┐
│ BoundedQueue          │
│ <ProfileEvent>        │
└──────────┬───────────┘
           │
           │ consumer thread
           ▼
┌──────────────────────┐
│ Event Processing     │
│ Serializer / Sink    │
└──────────────────────┘
```

---

## Key Concepts Practiced

This notebook introduces several important systems-programming concepts:

```text
producer-consumer architecture
shared mutable state
mutex
condition variable
bounded buffering
backpressure
thread lifecycle
std::jthread
std::stop_token
graceful shutdown
```

These concepts will later be reused when the queue feeds the TCP transport layer.

---

## Completion Criteria

This notebook is complete when:

```text
[ ] BoundedQueue<T> is implemented
[ ] Queue capacity is enforced
[ ] Producer waits when the queue is full
[ ] Consumer waits when the queue is empty
[ ] Multiple threads can safely access the queue
[ ] SyntheticEventSource runs in a producer thread
[ ] Event processing runs in a consumer thread
[ ] std::jthread is used for thread lifecycle management
[ ] The queue supports graceful shutdown
[ ] All events are processed without loss or deadlock
[ ] The implementation builds successfully with CMake
```

Once this pipeline is working, the next stage will add batching and prepare the event stream for network transport.

## Standalone Setup

This notebook is designed to run independently in a fresh Google Colab runtime.

The setup installs and verifies the minimal development tools required for the thread-safe C++20 pipeline.

Required tools:

- GCC / G++ with C++20 support
- CMake
- Ninja
- Git
- POSIX thread support

This notebook does not require a GPU.

The focus is on host-side systems programming:

- multithreading
- synchronization
- bounded queues
- producer-consumer execution
- thread lifecycle management
- graceful shutdown

In [1]:
!sudo apt-get update -qq

!sudo apt-get install -y \
    build-essential \
    g++ \
    cmake \
    ninja-build \
    git

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.10ubuntu1).
g++ is already the newest version (4:13.2.0-7ubuntu1).
g++ set to manually installed.
cmake is already the newest version (3.28.3-1build7).
git is already the newest version (1:2.43.0-1ubuntu7.3).
The following NEW packages will be installed:
  ninja-build
0 upgraded, 1 newly installed, 0 to remove and 64 not upgraded.
Need to get 129 kB of archives.
After this operation, 364 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 ninja-build amd64 1.11.1-2 [129 kB]
Fetched 129 kB in 0s (644 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so

In [2]:
!echo "=== Compiler ==="
!g++ --version | head -n 1

!echo ""
!echo "=== CMake ==="
!cmake --version | head -n 1

!echo ""
!echo "=== Ninja ==="
!ninja --version

=== Compiler ===
g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0

=== CMake ===
cmake version 3.31.10

=== Ninja ===
1.11.1


In [3]:
!rm -rf distributed-profiler

!mkdir -p distributed-profiler/include
!mkdir -p distributed-profiler/src
!mkdir -p distributed-profiler/tests
!mkdir -p distributed-profiler/benchmarks

## Prerequisite Core Components

This notebook builds on the core data model and interfaces introduced in the previous notebook.

To keep the notebook fully standalone, the required source files are recreated below.

The reused components are:

```text
include/
├── profile_event.hpp
├── event_source.hpp
├── serializer.hpp
├── data_sink.hpp
├── synthetic_event_source.hpp
├── simple_text_serializer.hpp
└── console_sink.hpp

src/
├── synthetic_event_source.cpp
├── simple_text_serializer.cpp
└── console_sink.cpp

In [4]:
%%writefile distributed-profiler/include/profile_event.hpp

#pragma once

#include <cstdint>
#include <string>
#include <variant>

struct KernelEvent {
    std::uint64_t timestamp_ns;
    std::uint64_t duration_ns;
    std::uint32_t device_id;
    std::string kernel_name;
};

struct MemoryEvent {
    std::uint64_t timestamp_ns;
    std::uint32_t device_id;
    std::uint64_t bytes;
    std::string operation;
};

using ProfileEvent =
    std::variant<KernelEvent, MemoryEvent>;

Writing distributed-profiler/include/profile_event.hpp


In [5]:
%%writefile distributed-profiler/include/event_source.hpp

#pragma once

#include "profile_event.hpp"

#include <optional>

class IEventSource {
public:
    virtual ~IEventSource() = default;

    virtual std::optional<ProfileEvent>
    next_event() = 0;
};

Writing distributed-profiler/include/event_source.hpp


In [6]:
%%writefile distributed-profiler/include/serializer.hpp

#pragma once

#include "profile_event.hpp"

#include <cstddef>
#include <vector>

class ISerializer {
public:
    virtual ~ISerializer() = default;

    virtual std::vector<std::byte>
    serialize(const ProfileEvent& event) const = 0;
};

Writing distributed-profiler/include/serializer.hpp


In [7]:
%%writefile distributed-profiler/include/data_sink.hpp

#pragma once

#include "profile_event.hpp"

class IDataSink {
public:
    virtual ~IDataSink() = default;

    virtual void write(
        const ProfileEvent& event
    ) = 0;
};

Writing distributed-profiler/include/data_sink.hpp


In [8]:
%%writefile distributed-profiler/include/synthetic_event_source.hpp

#pragma once

#include "event_source.hpp"

#include <cstddef>
#include <optional>
#include <vector>

class SyntheticEventSource : public IEventSource {
public:
    SyntheticEventSource();

    std::optional<ProfileEvent>
    next_event() override;

private:
    std::vector<ProfileEvent> events_;
    std::size_t current_index_;
};

Writing distributed-profiler/include/synthetic_event_source.hpp


In [9]:
%%writefile distributed-profiler/src/synthetic_event_source.cpp

#include "synthetic_event_source.hpp"

SyntheticEventSource::SyntheticEventSource()
    : current_index_(0) {

    events_.push_back(
        KernelEvent{
            1000,
            200,
            0,
            "gemm_kernel"
        }
    );

    events_.push_back(
        MemoryEvent{
            1500,
            0,
            4096,
            "H2D"
        }
    );

    events_.push_back(
        KernelEvent{
            2000,
            150,
            0,
            "softmax_kernel"
        }
    );

    events_.push_back(
        MemoryEvent{
            2300,
            0,
            8192,
            "D2H"
        }
    );
}

std::optional<ProfileEvent>
SyntheticEventSource::next_event() {

    if (current_index_ < events_.size()) {
        return events_[current_index_++];
    }

    return std::nullopt;
}

Writing distributed-profiler/src/synthetic_event_source.cpp


In [10]:
%%writefile distributed-profiler/include/simple_text_serializer.hpp

#pragma once

#include "serializer.hpp"

class SimpleTextSerializer : public ISerializer {
public:
    std::vector<std::byte>
    serialize(
        const ProfileEvent& event
    ) const override;
};

Writing distributed-profiler/include/simple_text_serializer.hpp


In [11]:
%%writefile distributed-profiler/src/simple_text_serializer.cpp

#include "simple_text_serializer.hpp"

#include <string>
#include <type_traits>
#include <variant>

std::vector<std::byte>
SimpleTextSerializer::serialize(
    const ProfileEvent& event
) const {

    std::string text;

    std::visit(
        [&](const auto& e) {
            using T =
                std::decay_t<decltype(e)>;

            if constexpr (
                std::is_same_v<T, KernelEvent>
            ) {
                text =
                    "KERNEL|" +
                    std::to_string(e.timestamp_ns) + "|" +
                    std::to_string(e.duration_ns) + "|" +
                    std::to_string(e.device_id) + "|" +
                    e.kernel_name;
            }
            else if constexpr (
                std::is_same_v<T, MemoryEvent>
            ) {
                text =
                    "MEMORY|" +
                    std::to_string(e.timestamp_ns) + "|" +
                    std::to_string(e.device_id) + "|" +
                    std::to_string(e.bytes) + "|" +
                    e.operation;
            }
        },
        event
    );

    std::vector<std::byte> bytes;
    bytes.reserve(text.size());

    for (char c : text) {
        bytes.push_back(
            static_cast<std::byte>(c)
        );
    }

    return bytes;
}

Writing distributed-profiler/src/simple_text_serializer.cpp


In [12]:
%%writefile distributed-profiler/include/console_sink.hpp

#pragma once

#include "data_sink.hpp"

class ConsoleSink : public IDataSink {
public:
    void write(
        const ProfileEvent& event
    ) override;
};

Writing distributed-profiler/include/console_sink.hpp


In [13]:
%%writefile distributed-profiler/src/console_sink.cpp

#include "console_sink.hpp"

#include <iostream>
#include <type_traits>
#include <variant>

void ConsoleSink::write(
    const ProfileEvent& event
) {

    std::visit(
        [](const auto& e) {
            using T =
                std::decay_t<decltype(e)>;

            if constexpr (
                std::is_same_v<T, KernelEvent>
            ) {
                std::cout
                    << "[KernelEvent] "
                    << "kernel=" << e.kernel_name
                    << ", device=" << e.device_id
                    << ", timestamp=" << e.timestamp_ns
                    << ", duration=" << e.duration_ns
                    << " ns\n";
            }
            else if constexpr (
                std::is_same_v<T, MemoryEvent>
            ) {
                std::cout
                    << "[MemoryEvent] "
                    << "operation=" << e.operation
                    << ", device=" << e.device_id
                    << ", bytes=" << e.bytes
                    << ", timestamp=" << e.timestamp_ns
                    << " ns\n";
            }
        },
        event
    );
}

Writing distributed-profiler/src/console_sink.cpp


## Step 1 — Implement a Bounded Queue

The first step is to implement a reusable thread-safe bounded queue.

The queue will sit between the producer and consumer threads:

```text
Producer Thread
      ↓
   push()
      ↓
BoundedQueue<T>
      ↓
    pop()
      ↓
Consumer Thread
```

The queue has a fixed capacity.

If the producer tries to insert data while the queue is full, it must wait until the consumer removes an item.

If the consumer tries to remove data while the queue is empty, it must wait until the producer inserts a new item.

The initial queue will provide three main operations:

```cpp
push(T item)
pop()
close()
```

- `push()` inserts an item into the queue.
- `pop()` removes and returns the next available item.
- `close()` indicates that no additional items will be produced and allows waiting consumers to exit cleanly after all remaining items have been processed.

The queue will internally use:

```text
std::deque<T>
std::mutex
std::condition_variable
bool closed_
std::size_t capacity_
```

The main goals of this step are:

- Keep shared queue state thread-safe.
- Enforce a fixed queue capacity.
- Block producers when the queue is full.
- Block consumers when the queue is empty.
- Support graceful shutdown.
- Make the queue reusable for different data types through `template<typename T>`.

In this project, the queue will later be instantiated as:

```cpp
BoundedQueue<ProfileEvent>
```

The expected behavior is:

```text
Producer faster than consumer
        ↓
Queue becomes full
        ↓
Producer waits
        ↓
Consumer removes an event
        ↓
Space becomes available
        ↓
Producer continues
```

This bounded queue provides the first backpressure mechanism in the profiling pipeline.

In [14]:
%%writefile distributed-profiler/include/bounded_queue.hpp

#pragma once

#include <condition_variable>
#include <cstddef>
#include <deque>
#include <mutex>
#include <optional>
#include <utility>

template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity)
        : capacity_(capacity),
          closed_(false) {}

    bool push(T item) {
        std::unique_lock<std::mutex> lock(mutex_);

        not_full_.wait( // predidate, if not closed or there is space, mutex_ will be returned and data will come back
            lock,
            [this]() {
                return queue_.size() < capacity_ || closed_;
            }
        );

        if (closed_) {
            return false;
        }

        queue_.push_back(std::move(item));

        lock.unlock();

        not_empty_.notify_one();

        return true;
    }

    std::optional<T> pop() {
        std::unique_lock<std::mutex> lock(mutex_);

        not_empty_.wait(
            lock,
            [this]() {
                return !queue_.empty() || closed_;
            }
        );

        if (queue_.empty() && closed_) {
            return std::nullopt;
        }

        T item = std::move(queue_.front());
        queue_.pop_front();

        lock.unlock();

        not_full_.notify_one();

        return item;
    }

    void close() {
        {
            std::lock_guard<std::mutex> lock(mutex_);
            closed_ = true;
        }

        not_empty_.notify_all();
        not_full_.notify_all();
    }

private:
    std::deque<T> queue_;

    std::size_t capacity_;

    std::mutex mutex_;

    std::condition_variable not_empty_;
    std::condition_variable not_full_;

    bool closed_;
};

Writing distributed-profiler/include/bounded_queue.hpp


## Step 2 — Validate the Bounded Queue

Before connecting `BoundedQueue<T>` to the profiling pipeline, the queue should first be validated independently with a simple data type.

The goal of this step is to confirm that the queue correctly handles:

- fixed capacity
- producer blocking when the queue is full
- consumer blocking when the queue is empty
- wake-up notifications between producer and consumer
- graceful shutdown through `close()`

A simple `BoundedQueue<int>` will be used for this validation.

The test will use:

```text
capacity = 2
```

and two threads:

```text
Producer Thread
      ↓
 push(1)
 push(2)
 push(3)
      ↓
BoundedQueue<int>
      ↓
 pop()
      ↓
Consumer Thread
```

The consumer will intentionally process items more slowly than the producer.

This should cause the queue to become full:

```text
Producer pushes 1
        ↓
Producer pushes 2
        ↓
Queue is full
        ↓
Producer tries to push 3
        ↓
Producer waits
```

When the consumer removes an item:

```text
Consumer pops 1
        ↓
Queue has available space
        ↓
not_full_.notify_one()
        ↓
Producer wakes up
        ↓
Producer pushes 3
```

This verifies that the bounded queue provides backpressure correctly.

The test will also validate queue shutdown.

After the producer finishes inserting all items, it will call:

```cpp
queue.close();
```

The consumer should continue processing any items that are still buffered.

Once the queue is both:

```text
closed
AND
empty
```

`pop()` should return:

```cpp
std::nullopt
```

and the consumer thread should exit normally.

The expected shutdown sequence is:

```text
Producer finishes
        ↓
queue.close()
        ↓
Consumer wakes up
        ↓
Remaining items are consumed
        ↓
Queue becomes empty
        ↓
pop() returns std::nullopt
        ↓
Consumer exits
```

This test isolates the synchronization behavior of `BoundedQueue<T>` before introducing `ProfileEvent`, `SyntheticEventSource`, serialization, or other profiling-specific components.

In [15]:
%%writefile distributed-profiler/tests/test_bounded_queue.cpp

#include "bounded_queue.hpp"

#include <chrono>
#include <iostream>
#include <thread>

using namespace std::chrono_literals;

int main() {
    // Create a bounded queue with capacity = 2.
    BoundedQueue<int> queue(2);

    // Create a producer thread.
    //
    // Producer behavior:
    // - push 1
    // - print "Producer pushed 1"
    // - push 2
    // - print "Producer pushed 2"
    // - push 3
    // - print "Producer pushed 3"
    // - call queue.close()
    //
    // Important:
    // push(3) should eventually block if the queue is full.
    std::thread producer([&queue]()
    {
        queue.push(1);
        std::cout << "Producer pushed 1" << std::endl;

        queue.push(2);
        std::cout << "Producer pushed 2"<< std::endl;

        std::cout << "Producer trying to push 3..."<<std::endl;

        queue.push(3);
        std::cout << "Producer pushed 3"<<std::endl;

        queue.close();
    });


    // Create a consumer thread.
    //
    // Consumer behavior:
    // - sleep briefly before starting
    // - repeatedly call queue.pop()
    // - if pop() returns an item:
    //       print the value
    //       sleep for a short period to simulate slow processing
    // - if pop() returns std::nullopt:
    //       exit the loop
    std::thread consumer([&queue]() {
        std::this_thread::sleep_for(1s);


        while(true){
            auto item = queue.pop();

            if(!item.has_value()){
                break;
            }

            std::cout<<"Consumer popped "<< *item<< std::endl;

            std::this_thread::sleep_for(50ms);
        }
    });


    // Wait for both threads to finish.
    producer.join();
    consumer.join();



    std::cout << "BoundedQueue test finished\n";

    return 0;
}

Writing distributed-profiler/tests/test_bounded_queue.cpp


In [16]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -pthread \
    -Idistributed-profiler/include \
    distributed-profiler/tests/test_bounded_queue.cpp \
    -o test_bounded_queue

In [17]:
!./test_bounded_queue

Producer pushed 1
Producer pushed 2
Producer trying to push 3...
Consumer popped 1
Producer pushed 3
Consumer popped 2
Consumer popped 3
BoundedQueue test finished


## Step 3 — Build an Asynchronous Profiling Event Pipeline

Now that `BoundedQueue<T>` has been validated independently, the next step is to integrate it with the profiling data model developed earlier.

The goal is to replace the simple integer producer-consumer test with a real profiling-event pipeline.

The new architecture is:

```text
SyntheticEventSource
        ↓
   Producer Thread
        ↓
BoundedQueue<ProfileEvent>
        ↓
   Consumer Thread
        ↓
Serializer / ConsoleSink
```

The producer thread is responsible for retrieving profiling events from `SyntheticEventSource`.

Its behavior is:

```text
call next_event()
        ↓
event available?
   ├── yes → push event into queue
   │
   └── no  → close queue and exit
```

Conceptually:

```cpp
while (auto event = source.next_event()) {
    queue.push(...);
}

queue.close();
```

The consumer thread is responsible for removing profiling events from the queue and processing them.

Its behavior is:

```text
call pop()
    ↓
event available?
   ├── yes → process event
   │
   └── no  → queue is closed and empty
             → exit
```

Conceptually:

```cpp
while (auto event = queue.pop()) {
    // process ProfileEvent
}
```

The consumer can initially send each event to:

```text
ConsoleSink
```

and optionally serialize it using:

```text
SimpleTextSerializer
```

This creates the first real asynchronous profiling pipeline in the project.

Instead of producing and processing an event in the same execution path:

```text
Source
  ↓
Event
  ↓
Process immediately
```

the producer and consumer now execute independently:

```text
Producer                     Consumer
   │                            │
   │ generate event             │
   ▼                            │
BoundedQueue                    │
   │                            │
   └───────────────────────────→│
                                │ process event
```

The queue acts as the synchronization boundary between the two threads.

If the producer becomes faster than the consumer:

```text
Producer faster
      ↓
Queue fills
      ↓
Producer blocks
      ↓
Consumer processes events
      ↓
Space becomes available
      ↓
Producer continues
```

If the consumer becomes faster than the producer:

```text
Consumer reaches empty queue
      ↓
Consumer waits
      ↓
Producer pushes new event
      ↓
Consumer is notified
      ↓
Consumer continues
```

This step validates that the profiling data abstraction and the thread-safe queue can work together correctly.

### Main Goals

- Use `BoundedQueue<ProfileEvent>` instead of `BoundedQueue<int>`.
- Run `SyntheticEventSource` inside a producer thread.
- Push generated `ProfileEvent` objects into the queue.
- Run event processing inside a separate consumer thread.
- Process events using `ConsoleSink`.
- Close the queue when the event source is exhausted.
- Allow the consumer to process all buffered events before exiting.
- Verify that all profiling events are transferred without loss.

### Expected Data Flow

```text
SyntheticEventSource
        ↓
KernelEvent / MemoryEvent
        ↓
ProfileEvent
        ↓
queue.push()
        ↓
BoundedQueue<ProfileEvent>
        ↓
queue.pop()
        ↓
ProfileEvent
        ↓
ConsoleSink
```

After this step, the project will have a working multi-threaded producer-consumer profiling pipeline.

The next step will replace basic `std::thread` lifecycle management with modern C++20 thread management using `std::jthread` and `std::stop_token`.

In [18]:
%%writefile distributed-profiler/tests/test_async_profile_pipeline.cpp

#include "bounded_queue.hpp"
#include "console_sink.hpp"
#include "profile_event.hpp"
#include "synthetic_event_source.hpp"

#include <chrono>
#include <iostream>
#include <thread>

using namespace std::chrono_literals;

int main() {
    // Create:
    // - a SyntheticEventSource
    // - a ConsoleSink
    // - a BoundedQueue<ProfileEvent> with a small capacity,
    //   for example capacity = 2
    SyntheticEventSource source;
    ConsoleSink sink;

    BoundedQueue<ProfileEvent> queue(2);


    // Create the producer thread.
    //
    // Producer logic:
    //
    // while source.next_event() returns an event:
    //     push the event into the queue
    //     optionally print that an event was pushed
    //
    // when the source is exhausted:
    //     close the queue
    std::thread producer([&](){
        while (auto event = source.next_event()){
            queue.push(std::move(*event));

            std::cout <<  "Producer pushed one event" << std::endl;
        }

        queue.close();

        std::cout<< "Producer finished and closed the queue" << std::endl;
    });

    // Create the consumer thread.
    //
    // Consumer logic:
    //
    // repeatedly call queue.pop()
    //
    // if an event is returned:
    //     send it to ConsoleSink
    //     optionally sleep briefly to simulate slower processing
    //
    // if std::nullopt is returned:
    //     exit the loop

    std::thread consumer([&](){
        while (auto event = queue.pop()){
            sink.write(*event);

            std::this_thread::sleep_for(500ms);
        }

        std::cout << "Consumer finished" <<std::endl;
    });


    // Wait for the producer and consumer threads to finish.

    producer.join();
    consumer.join();


    std::cout
        << "Async profiling pipeline finished\n";

    return 0;
}

Writing distributed-profiler/tests/test_async_profile_pipeline.cpp


In [19]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -pthread \
    -Idistributed-profiler/include \
    distributed-profiler/src/synthetic_event_source.cpp \
    distributed-profiler/src/console_sink.cpp \
    distributed-profiler/tests/test_async_profile_pipeline.cpp \
    -o test_async_profile_pipeline

In [20]:
!./test_async_profile_pipeline

[KernelEvent] kernel=gemm_kernel, device=0, timestamp=1000, duration=200 ns
Producer pushed one event
Producer pushed one event
Producer pushed one event
[MemoryEvent] operation=H2D, device=0, bytes=4096, timestamp=1500 ns
Producer pushed one event
Producer finished and closed the queue
[KernelEvent] kernel=softmax_kernel, device=0, timestamp=2000, duration=150 ns
[MemoryEvent] operation=D2H, device=0, bytes=8192, timestamp=2300 ns
Consumer finished
Async profiling pipeline finished


## Step 4 — Modern Thread Lifecycle Management with `std::jthread` and `std::stop_token`

The asynchronous profiling pipeline currently uses `std::thread`.

The existing pattern is:

```cpp
std::thread producer(...);
std::thread consumer(...);

producer.join();
consumer.join();
```

This works correctly, but thread lifetime must be managed manually.

If a `std::thread` object is destroyed while it is still joinable, the program will call:

```text
std::terminate()
```

Therefore, every `std::thread` must eventually be either:

```cpp
join()
```

or:

```cpp
detach()
```

For this project, the next step is to use the C++20 threading model based on:

```cpp
std::jthread
std::stop_token
```

---

### Why Use `std::jthread`?

`std::jthread` was introduced in C++20.

It is similar to `std::thread`, but it provides two important improvements:

```text
std::jthread
   ├── automatically joins when destroyed
   └── supports cooperative cancellation
```

With `std::thread`, we normally write:

```cpp
std::thread worker(...);

worker.join();
```

With `std::jthread`, explicit joining is usually unnecessary:

```cpp
std::jthread worker(...);
```

When the `std::jthread` object goes out of scope, its destructor automatically waits for the thread to finish.

Conceptually:

```text
std::thread
    ↓
manual join()

std::jthread
    ↓
automatic join
```

This reduces the risk of forgetting to join a thread.

---

## Cooperative Cancellation

Another important feature of `std::jthread` is built-in stop support.

A thread function can receive a:

```cpp
std::stop_token
```

For example:

```cpp
std::jthread worker([](std::stop_token stop_token) {
    while (!stop_token.stop_requested()) {
        // perform work
    }
});
```

The thread periodically checks:

```cpp
stop_token.stop_requested()
```

to determine whether another part of the program has requested it to stop.

The controlling thread can request cancellation using:

```cpp
worker.request_stop();
```

The control flow becomes:

```text
Main Thread
    ↓
request_stop()
    ↓
stop request becomes visible
    ↓
Worker Thread
    ↓
stop_token.stop_requested()
    ↓
true
    ↓
exit loop
```

---

## Cooperative Means Cooperative

`std::stop_token` does not forcibly terminate a thread.

Instead, it provides a communication mechanism.

The worker thread must explicitly check the stop request:

```cpp
if (stop_token.stop_requested()) {
    break;
}
```

Therefore:

```text
request_stop()
```

does not mean:

```text
kill the thread immediately
```

It means:

```text
ask the thread to stop
```

The thread decides when it is safe to exit.

This is useful because forced thread termination could leave:

- mutexes locked
- queues in an inconsistent state
- resources unreleased
- partially written profiling data

Cooperative cancellation allows the worker to exit cleanly.

---

## Updated Profiling Pipeline

The architecture remains the same:

```text
SyntheticEventSource
        ↓
   Producer jthread
        ↓
BoundedQueue<ProfileEvent>
        ↓
   Consumer jthread
        ↓
ConsoleSink
```

The main difference is thread lifecycle management.

Instead of:

```cpp
std::thread producer(...);
std::thread consumer(...);

producer.join();
consumer.join();
```

we will use:

```cpp
std::jthread producer(...);
std::jthread consumer(...);
```

and allow the `std::jthread` objects to automatically join when they leave scope.

---

## Adding `std::stop_token`

The producer and consumer thread functions can receive a stop token:

```cpp
std::jthread producer(
    [&](std::stop_token stop_token) {
        // producer logic
    }
);
```

and:

```cpp
std::jthread consumer(
    [&](std::stop_token stop_token) {
        // consumer logic
    }
);
```

Inside the thread, the token can be checked:

```cpp
if (stop_token.stop_requested()) {
    break;
}
```

For example:

```text
Producer loop
    ↓
check stop request
    ↓
generate event
    ↓
push event
    ↓
repeat
```

---

## Important Limitation

A stop request does not automatically interrupt the current implementation of:

```cpp
queue.push()
```

or:

```cpp
queue.pop()
```

because these functions may block inside:

```cpp
std::condition_variable::wait()
```

For example:

```text
Consumer
   ↓
queue.pop()
   ↓
queue empty
   ↓
condition_variable::wait()
```

If another thread calls:

```cpp
consumer.request_stop();
```

the stop request becomes visible through the `stop_token`, but the blocked `condition_variable::wait()` does not automatically wake up.

Therefore:

```text
request_stop()
```

and:

```text
condition_variable notification
```

are separate mechanisms.

This is an important concurrency design issue.

For now, this step will focus on understanding:

- `std::jthread`
- automatic joining
- `std::stop_token`
- `request_stop()`
- cooperative cancellation

Later, the queue can be extended to make blocking operations cancellation-aware.

---

## Goals of This Step

By the end of this section, the following questions should be answerable:

- What is the difference between `std::thread` and `std::jthread`?
- Why does `std::thread` require `join()`?
- Why can `std::jthread` usually avoid explicit `join()`?
- What is a `std::stop_token`?
- What does `request_stop()` do?
- Why is cancellation called cooperative?
- Why does `request_stop()` not necessarily wake a thread blocked in `condition_variable::wait()`?
- How can modern C++20 thread management improve the profiling pipeline?

In [21]:
%%writefile distributed-profiler/tests/test_jthread_pipeline.cpp

#include "bounded_queue.hpp"
#include "console_sink.hpp"
#include "profile_event.hpp"
#include "synthetic_event_source.hpp"

#include <chrono>
#include <iostream>
#include <stop_token>
#include <thread>

using namespace std::chrono_literals;

int main() {
    SyntheticEventSource source;
    ConsoleSink sink;

    BoundedQueue<ProfileEvent> queue(2);

    // Create a producer using std::jthread.
    //
    // The lambda should receive:
    //
    // std::stop_token stop_token
    //
    // Producer logic:
    //
    // while there is an event from source:
    //     check whether stop was requested
    //     if yes:
    //         break
    //
    //     otherwise:
    //         push the event into the queue
    //         print a message
    //
    // when producer exits:
    //     close the queue

    std::jthread producer(
        [&](std::stop_token stop_token){
            while (auto event = source.next_event()){
                if (stop_token.stop_requested()){
                    std::cout << "Producer received stop request\n";
                    break;
                }

                queue.push(std::move(*event));
                std::cout << "Producer pushed one event\n";
            }
            queue.close();
            std::cout << "Producer finished and closed the queue\n";
        }
    );


    // Create a consumer using std::jthread.
    //
    // The lambda should also receive:
    //
    // std::stop_token stop_token
    //
    // Consumer logic:
    //
    // repeatedly:
    //     check whether stop was requested
    //
    //     call queue.pop()
    //
    //     if queue.pop() returns std::nullopt:
    //         exit
    //
    //     otherwise:
    //         write the event using ConsoleSink
    //         sleep briefly to simulate processing
    std::jthread consumer(
        [&](std::stop_token stop_token){
            while(true){
                if(stop_token.stop_requested()){
                    std::cout << "Consumer received stop request\n";
                    break;
                }
                auto event = queue.pop();
                if (!event.has_value()) {
                    break;
                }

                sink.write(*event);

                std::this_thread::sleep_for(500ms);
            }
            std::cout << "Consumer finished\n";
        }
    );


    // Let the pipeline run for a short time.
    std::this_thread::sleep_for(1s);


    // Request the threads to stop.
    //
    producer.request_stop();
    consumer.request_stop();

    // Do NOT explicitly call join().
    //
    // Observe what happens when producer and consumer
    // go out of scope at the end of main().
    //
    // std::jthread will automatically join.

    std::cout << "Main thread requested stop\n";
    std::cout << "Main thread is finishing\n";

    return 0;
}

Writing distributed-profiler/tests/test_jthread_pipeline.cpp


In [22]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -pthread \
    -Idistributed-profiler/include \
    distributed-profiler/src/synthetic_event_source.cpp \
    distributed-profiler/src/console_sink.cpp \
    distributed-profiler/tests/test_jthread_pipeline.cpp \
    -o test_jthread_pipeline

In [23]:
!./test_jthread_pipeline

Producer pushed one event
Producer pushed one event
[KernelEvent] kernel=gemm_kernel, device=0, timestamp=1000, duration=200 ns
Producer pushed one event
[MemoryEvent] operation=H2D, device=0, bytes=4096, timestamp=1500 ns
Producer pushed one event
Producer finished and closed the queue
Main thread requested stop
Main thread is finishing
Consumer received stop request
Consumer finished


## Step 5 — Make `BoundedQueue` Cancellation-Aware

The current `BoundedQueue<T>` supports blocking producer-consumer synchronization, but it does not yet respond directly to `std::stop_token`.

The problem appears when a thread is blocked inside:

```cpp
queue.push(...)
```

or:

```cpp
queue.pop()
```

For example, consider a consumer waiting on an empty queue:

```text
Consumer Thread
      ↓
queue.pop()
      ↓
queue is empty
      ↓
condition_variable::wait()
      ↓
blocked
```

Now another thread calls:

```cpp
consumer.request_stop();
```

The stop request becomes visible through the consumer's `std::stop_token`.

However, the consumer is still blocked inside:

```cpp
condition_variable::wait()
```

and may not get a chance to check:

```cpp
stop_token.stop_requested()
```

Therefore, simply checking the stop token before calling `pop()` is not sufficient.

---

## Goal

The goal of this step is to make the queue itself aware of cancellation.

The queue API will conceptually become:

```cpp
bool push(
    T item,
    std::stop_token stop_token
);
```

and:

```cpp
std::optional<T> pop(
    std::stop_token stop_token
);
```

This allows blocking queue operations to react directly to a stop request.

---

## Desired Producer Behavior

The producer may be blocked because the queue is full:

```text
Producer
    ↓
push(event)
    ↓
queue full
    ↓
wait
```

The producer should wake up when any of the following becomes true:

```text
1. queue has available space
2. queue is closed
3. stop is requested
```

Conceptually:

```text
wait until:

queue has space
OR
queue is closed
OR
stop requested
```

If cancellation is requested, `push()` should return:

```cpp
false
```

so that the producer can exit cleanly.

---

## Desired Consumer Behavior

The consumer may be blocked because the queue is empty:

```text
Consumer
    ↓
pop()
    ↓
queue empty
    ↓
wait
```

The consumer should wake up when any of the following becomes true:

```text
1. queue contains data
2. queue is closed
3. stop is requested
```

Conceptually:

```text
wait until:

queue has data
OR
queue is closed
OR
stop requested
```

If cancellation is requested, `pop()` should return:

```cpp
std::nullopt
```

so that the consumer can exit.

---

## Why `std::condition_variable_any`?

The current queue uses:

```cpp
std::condition_variable
```

For cancellation-aware waiting, C++20 provides stop-token-aware waiting support through:

```cpp
std::condition_variable_any
```

This allows a wait operation to receive a:

```cpp
std::stop_token
```

Conceptually:

```cpp
condition.wait(
    lock,
    stop_token,
    predicate
);
```

Now the wait can end because:

```text
predicate becomes true
```

or because:

```text
stop is requested
```

This connects the thread cancellation mechanism directly with the blocking queue operation.

---

## Updated Queue Semantics

The new `push()` behavior will be:

```text
push(item, stop_token)
        ↓
queue has space?
   ├── yes → insert item
   │
   └── no → wait
              ↓
       wake because:
       - space available
       - queue closed
       - stop requested
```

Return value:

```text
true
    → item was successfully inserted

false
    → queue was closed
      OR
      stop was requested
```

The new `pop()` behavior will be:

```text
pop(stop_token)
      ↓
queue contains data?
   ├── yes → remove and return item
   │
   └── no → wait
              ↓
       wake because:
       - data available
       - queue closed
       - stop requested
```

Return value:

```text
ProfileEvent
    → an item was successfully removed

std::nullopt
    → queue is closed and empty
      OR
      stop was requested
```

---

## Cancellation-Aware Pipeline

After this change, the pipeline becomes:

```text
SyntheticEventSource
        ↓
Producer jthread
        ↓
push(event, stop_token)
        ↓
BoundedQueue<ProfileEvent>
        ↓
pop(stop_token)
        ↓
Consumer jthread
        ↓
ConsoleSink
```

Now cancellation can propagate directly into blocking queue operations.

---

## Important Difference

Previously:

```text
request_stop()
     ↓
stop token becomes true
     ↓
thread may still remain blocked
```

After this change:

```text
request_stop()
     ↓
stop-aware wait wakes
     ↓
push/pop detects cancellation
     ↓
thread exits
```

This is a more robust design for asynchronous systems.

---

## First Validation Test

Before reconnecting the queue to the full profiling pipeline, the cancellation behavior should be tested independently.

The first test should use this situation:

```text
BoundedQueue<int>
      ↓
empty queue

Consumer jthread
      ↓
queue.pop(stop_token)
      ↓
blocked
```

Then the main thread will call:

```cpp
consumer.request_stop();
```

Expected behavior:

```text
consumer blocked
      ↓
request_stop()
      ↓
consumer wakes
      ↓
pop() returns std::nullopt
      ↓
consumer exits
```

Importantly, this test should work without calling:

```cpp
queue.close();
```

The stop request itself should be enough to wake the blocked consumer.

---

## Goals of This Step

By the end of this section, the following questions should be answerable:

- Why is checking `stop_requested()` before `queue.pop()` insufficient?
- Why can a thread remain blocked after `request_stop()`?
- What is the difference between `std::condition_variable` and `std::condition_variable_any` in this design?
- How can a blocking queue react directly to `std::stop_token`?
- What should `push()` return when cancellation occurs?
- What should `pop()` return when cancellation occurs?
- How does cancellation propagate through a producer-consumer pipeline?
- Why is cooperative cancellation useful in production asynchronous systems?

In [24]:
%%writefile distributed-profiler/include/bounded_queue.hpp

#pragma once

#include <condition_variable>
#include <cstddef>
#include <deque>
#include <mutex>
#include <optional>
#include <stop_token>
#include <utility>

template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity)
        : capacity_(capacity),
          closed_(false) {}

    bool push(
        T item,
        std::stop_token stop_token
    ) {
        // Acquire the mutex using std::unique_lock.
        std::unique_lock<std::mutex> lock(mutex_);


        // Wait until one of these happens:
        //
        // - queue has available space
        // - queue is closed
        // - stop is requested
        //
        // Use the stop-token-aware wait overload of
        // std::condition_variable_any.
        bool ready = not_full_.wait(
            lock,
            stop_token,
            [this]() {
                return queue_.size() < capacity_ || closed_;
            }
        );

        // After wait returns, check whether:
        //
        // - stop was requested
        // - queue was closed
        //
        // If either is true:
        //     return false;
        if (!ready || stop_token.stop_requested() || closed_) {
            return false;
        }


        // Move the item into queue_.
        queue_.push_back(std::move(item));


        // Unlock before notification.
        lock.unlock();


        // Notify one waiting consumer.
        not_empty_.notify_one();


        // Return true to indicate successful push.
        return true;

    }


    std::optional<T> pop(
        std::stop_token stop_token
    ) {
        // Acquire the mutex using std::unique_lock.
        std::unique_lock<std::mutex> lock(mutex_);


        // Wait until one of these happens:
        //
        // - queue contains an item
        // - queue is closed
        // - stop is requested
        //
        // Again use the stop-token-aware wait overload.
        bool ready = not_empty_.wait(
            lock,
            stop_token,
            [this]() {
                return !queue_.empty() || closed_;
            }
        );


        // If stop was requested:
        //     return std::nullopt;
        if (!ready || stop_token.stop_requested()) {
            return std::nullopt;
        }


        // If queue is empty and closed:
        //     return std::nullopt;
        if (queue_.empty() && closed_) {
            return std::nullopt;
        }


        // Move the front item out of the queue.
        T item = std::move(queue_.front());


        // Remove the front item.
        queue_.pop_front();

        // Unlock before notification.
        lock.unlock();


        // Notify one waiting producer.
        not_full_.notify_one();


        // Return the item.
        return item;

    }


    void close() {
        {
            std::lock_guard<std::mutex> lock(mutex_);
            closed_ = true;
        }

        not_empty_.notify_all();
        not_full_.notify_all();
    }

private:
    std::deque<T> queue_;
    std::size_t capacity_;

    std::mutex mutex_;

    // Change these from std::condition_variable
    // to std::condition_variable_any
    //
    // Why?
    // Because condition_variable_any provides the
    // C++20 stop_token-aware wait overload.

    std::condition_variable_any not_empty_;
    std::condition_variable_any not_full_;

    bool closed_;
};

Overwriting distributed-profiler/include/bounded_queue.hpp


In [25]:
%%writefile distributed-profiler/tests/test_queue_cancellation.cpp

#include "bounded_queue.hpp"

#include <chrono>
#include <iostream>
#include <stop_token>
#include <thread>

using namespace std::chrono_literals;

int main() {
    BoundedQueue<int> queue(2);

    std::jthread consumer(
        [&](std::stop_token stop_token) {
            std::cout << "Consumer: calling pop() and waiting...\n";

            auto item = queue.pop(stop_token);

            if (!item.has_value()) {
                std::cout << "Consumer: pop() returned std::nullopt\n";
                std::cout << "Consumer: exiting\n";
                return;
            }

            std::cout
                << "Consumer received item: "
                << *item
                << '\n';
        }
    );

    std::this_thread::sleep_for(1s);

    std::cout << "Main: requesting consumer stop\n";

    consumer.request_stop();

    std::cout << "Main: stop request sent\n";

    return 0;
}

Writing distributed-profiler/tests/test_queue_cancellation.cpp


In [26]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -pthread \
    -Idistributed-profiler/include \
    distributed-profiler/tests/test_queue_cancellation.cpp \
    -o test_queue_cancellation

In [27]:
!./test_queue_cancellation

Consumer: calling pop() and waiting...
Main: requesting consumer stop
Main: stop request sent
Consumer: pop() returned std::nullopt
Consumer: exiting
